In [1]:
from utils import *

In [2]:
EJ=8.9
EC=2.5
EL=0.5
g_strength = 0.3

E_osc = 3

qubit_level = 8
osc_level = 50

In [3]:
qbt = scqubits.Fluxonium(EJ=EJ,EC=EC,EL=EL,flux=0,cutoff=30,truncated_dim=qubit_level)
osc = scqubits.Oscillator(E_osc=E_osc,truncated_dim=osc_level)
hilbertspace = scqubits.HilbertSpace([qbt, osc])
hilbertspace.add_interaction(g_strength=g_strength,op1=qbt.n_operator,op2=osc.creation_operator,add_hc=True)
hilbertspace.generate_lookup()
product_to_dressed = generate_single_mapping(hilbertspace.hamiltonian())
# plot_specturum(qbt, osc, hilbertspace)

max overlap^2 0.015315063260411915 below threshold for dressed state 148 with eval 66.01994392158106


In [4]:
a = hilbertspace.op_in_dressed_eigenbasis(op=osc.annihilation_operator)
a = qutip.Qobj(a[:, :])
 
(evals,) = hilbertspace["evals"]
diag_dressed_hamiltonian = (
        2 * np.pi * qutip.Qobj(np.diag(evals),
        dims=[hilbertspace.subsystem_dims] * 2)
)
diag_dressed_hamiltonian = qutip.Qobj(diag_dressed_hamiltonian[:, :])

leakage_dressed_state_osc_0 = product_to_dressed[(0,0)]
leakage_dressed_state_osc_1 = product_to_dressed[(0,1)]
w_d = transition_frequency(hilbertspace,leakage_dressed_state_osc_0,leakage_dressed_state_osc_1 )


amp = 0.0035
def square_cos(t,*args):
    cos = np.cos(w_d * 2*np.pi * t)
    return  2*np.pi *amp * cos

t_stop = 250 
def square_cos_with_ring_down(t,*args):
    if t > t_stop:
        return 0
    else:
        cos = np.cos(w_d * 2*np.pi * t)
        return  2*np.pi *amp * cos


H_with_drive = [
    diag_dressed_hamiltonian,
    [a+a.dag(), square_cos_with_ring_down]]

kappa = 0.07
decay_term = kappa*a

tot_time = 1000
tlist = np.linspace(0, tot_time, tot_time)[::4]



state_0_dressed = qutip.basis(hilbertspace.dimension, product_to_dressed[(1,0)])
state_1_dressed = qutip.basis(hilbertspace.dimension, product_to_dressed[(2,0)])
state_plus_dressed = (state_0_dressed + 1j * state_1_dressed).unit()
state_minus_dressed = (state_0_dressed - state_1_dressed).unit()
initial_states  = [state_0_dressed,state_1_dressed,state_plus_dressed,state_minus_dressed ]

# existing_chunk_num = 0
# for ini_state in initial_states:
#     existing_chunk_num = pack_mcsolve_chunks(H = H_with_drive,
#                     state0 = ini_state,
#                     tlist = tlist,
#                     c_ops  = [decay_term],
#                     ntraj = 500,
#                     existing_chunk_num = existing_chunk_num,
#                     chunk_size = 4)

# def pack_pkl_files_to_zip(zip_filename="mcsolve_input.zip"):
#     # Create a new ZIP file
#     with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
#         # Loop through all files in the current directory
#         for filename in os.listdir('.'):
#             # Check if the file is a .pkl file with an integer name
#             name, ext = os.path.splitext(filename)
#             if ext == '.pkl' and name.isdigit():
#                 # Add the file to the ZIP
#                 zipf.write(filename)
#                 # Delete the .pkl file
#                 os.remove(filename)
                
# pack_pkl_files_to_zip()

# Average results

In [5]:

# List of zip files containing the results
zip_files = [f"mcsolve_result_0.07_d0.0035_tomo_ring_down/result_{i}.zip" for i in range(500)]

# Divide the files into four equal parts
n_parts = 4
part_length = len(zip_files) // n_parts
zip_file_parts = [zip_files[i * part_length: (i + 1) * part_length] for i in range(n_parts)]

# Initialize an empty list to store the four results
results = []

# Merge the results for each part and append to the results list
for part in zip_file_parts:
    results.append(merge_results(part))


In [ ]:
# Unpickle the dictionary
with open('mcsolve_result_0.07_d0.0035_tomo_ring_down/averaged.pkl', 'wb') as f:
    pickle.dump(results,f)


# 1. Partial trace out qubit, truncate to 2 level, optimize a phase on the final state and get fidelity for each initial states